# 清洗与面板构建

展示真实数据经过清洗后生成的全球主面板、资源面板和中国省级面板。


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if ROOT.name == "notebooks" and (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError("无法定位仓库根目录，请从项目根目录或 notebooks/ 目录启动 Notebook。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def read_api(relpath: str):
    payload = read_json(ROOT / relpath)
    return payload.get("data", payload)

def show_image(relpath: str, width: int = 900):
    display(Image(filename=str(ROOT / relpath), width=width))

def require(*relpaths: str):
    missing = [path for path in relpaths if not (ROOT / path).exists()]
    if missing:
        raise FileNotFoundError(
            "缺少以下产物，请先运行 `PYTHONPATH=src python -m hdi.data.integrator` 和 "
            "`PYTHONPATH=src python -m hdi.analysis.competition`:\n- " + "\n- ".join(missing)
        )


In [ ]:
require(
    "data/processed/master_panel.parquet",
    "data/processed/resource_panel.parquet",
    "data/processed/china_panel.parquet",
)

master = pd.read_parquet(ROOT / "data/processed/master_panel.parquet")
resource = pd.read_parquet(ROOT / "data/processed/resource_panel.parquet")
china = pd.read_parquet(ROOT / "data/processed/china_panel.parquet")

display(pd.DataFrame([
    {"面板": "master_panel", "形状": master.shape, "国家数": master["iso3"].nunique(), "年份范围": f'{master["year"].min()}-{master["year"].max()}'},
    {"面板": "resource_panel", "形状": resource.shape, "国家数": resource["iso3"].nunique(), "年份范围": f'{resource["year"].min()}-{resource["year"].max()}'},
    {"面板": "china_panel", "形状": china.shape, "地区数": china["province"].nunique(), "年份范围": f'{china["year"].min()}-{china["year"].max()}'},
]))


In [ ]:
display(master.head())
display(china.head())
